# Agentic AI Research Agent
**Flow:** User Query → Tavily Web Search (5 sources) → LLM Synthesizer → Research Answer

In [1]:
!pip install -q langchain langgraph langchain-openai langchain-tavily tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 13.0 MB/s eta 0:00:00


In [ ]:
# ── CELL 1: Imports & API Keys ──────────────────────────────────────────────

import os
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START, END

os.environ["TAVILY_API_KEY"]  = "tvlyNc"
os.environ["GROQ_API_KEY"]    = "gsk_a6L"

In [12]:
# ── CELL 2: State ────────────────────────────────────────────────────────────
# Three fields travel through the graph:
#   query        — the original user question (set at entry, never changed)
#   web_results  — raw Tavily output, filled by web_search_node
#   final_answer — synthesized markdown answer, filled by synthesizer_node

class ResearchState(TypedDict):
    query:        str
    web_results:  str
    final_answer: str

In [13]:
# ── CELL 3: Tools & LLM ──────────────────────────────────────────────────────

# Tavily — returns top-5 web results as structured dicts
tavily = TavilySearch(max_results=5)

# Groq LLaMA via OpenAI-compatible endpoint (free tier)
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [14]:
# ── CELL 4: Node 1 — Web Search ──────────────────────────────────────────────
# Calls Tavily, formats the 5 results into a numbered text block,
# and stores it in state["web_results"].

def web_search_node(state: ResearchState) -> dict:

    raw = tavily.invoke({"query": state["query"]})

    raw = raw["results"]   # <-- this line is the fix

    # raw is a list of dicts: {url, content, title, score, ...}

    # Build a clean numbered block for the LLM prompt
    formatted_results = []

    for i, result in enumerate(raw, start=1):

        title   = result.get("title",   "No title")
        url     = result.get("url",     "")
        content = result.get("content", "").strip()

        formatted_results.append(
            f"[{i}] {title}\n"
            f"    URL: {url}\n"
            f"    {content}"
        )

    web_results_text = "\n\n".join(formatted_results)

    print(f"    Retrieved {len(raw)} sources.")

    return {"web_results": web_results_text}

In [15]:
# ── CELL 5: Node 2 — LLM Synthesizer ─────────────────────────────────────────
# Receives both the original query and the 5 web snippets.
# The system prompt instructs the LLM to blend web data with its
# own parametric knowledge into one well-structured answer.

SYSTEM_PROMPT = """\
You are an expert research assistant.

You will be given:
  1. A user research query.
  2. Five web search results (title, URL, snippet).

Your job:
  - Synthesize the web results with your own knowledge.
  - Write a clear, well-structured, comprehensive answer.
  - Cite sources inline using [1], [2], etc. where relevant.
  - Highlight any conflicting information across sources.
  - End with a "Sources" section listing all URLs.
  - Be factual. Do not hallucinate beyond the provided data.
"""


def synthesizer_node(state: ResearchState) -> dict:

    user_prompt = (
        f"Research Query: {state['query']}\n\n"
        f"Web Search Results:\n\n{state['web_results']}"
    )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]

    response = llm.invoke(messages)

    return {"final_answer": response.content}

In [16]:
# ── CELL 6: Build Graph ───────────────────────────────────────────────────────
#
#   START → web_search_node → synthesizer_node → END

workflow = StateGraph(ResearchState)

workflow.add_node("web_search",   web_search_node)
workflow.add_node("synthesizer",  synthesizer_node)

workflow.add_edge(START,          "web_search")
workflow.add_edge("web_search",   "synthesizer")
workflow.add_edge("synthesizer",  END)

agent = workflow.compile()

print("Graph compiled successfully.")

Graph compiled successfully.


In [17]:
# ── CELL 7: Run Function ──────────────────────────────────────────────────────

def research(query: str):
    """
    Run the research agent on a query.
    Prints a formatted answer with sources.
    """

    print(f"\n{'='*60}") # this is for making the output butiful
    print(f" QUERY: {query}")
    print(f"{'='*60}\n")

    result = agent.invoke({
        "query":        query,
        "web_results":  "",
        "final_answer": ""
    })

    print(f"\n{'─'*60}") # same butilful
    print(" RESEARCH ANSWER")
    print(f"{'─'*60}\n")
    print(result["final_answer"])
    print(f"\n{'='*60}\n")

    return result

In [18]:
# ── CELL 8: Test It ───────────────────────────────────────────────────────────

research("What are the latest developments in quantum computing in 2025?")


 QUERY: What are the latest developments in quantum computing in 2025?

[1/2] Searching the web...
    Retrieved 5 sources.
[2/2] Synthesizing answer...

────────────────────────────────────────────────────────────
 RESEARCH ANSWER
────────────────────────────────────────────────────────────

The latest developments in quantum computing in 2025 have been marked by significant advancements and investments in the field. According to [1], quantum networking has come into focus, with companies like IonQ and Cisco playing key roles. Additionally, Microsoft has announced plans to build a quantum center in Maryland, and Rigetti has secured two major contracts. 

The quantum computing market is expected to enable exponentially faster drug discovery, battery chemistry development, and other complex applications [2]. In terms of trends, Moody's has identified six key areas to watch in 2025, including more experiments with logical qubits, specialized hardware and software, and quantum networking

{'query': 'What are the latest developments in quantum computing in 2025?',
 'web_results': "[1] 2025 year in review: Quantum computing development accelerates\n    URL: https://www.constellationr.com/insights/news/2025-year-review-quantum-computing-development-accelerates\n    Quantum networking coming into focus with IonQ and Cisco playing roles. · Microsoft said it will build quantum center in Maryland. · Rigetti said it landed two\n\n[2] Quantum Computing Market 2025-2045: Technology, Trends ...\n    URL: https://www.idtechex.com/en/research-report/quantum-computing-market-2025/1053\n    The quantum computing market is pitched as enabling exponentially faster drug discovery, battery chemistry development, multi-variable logistics, vehicle\n\n[3] Quantum computing's six most important trends for 2025 - Moody's\n    URL: https://www.moodys.com/web/en/us/insights/quantum/quantum-computings-six-most-important-trends-for-2025.html\n    More experiments with logical qubits · More special

In [19]:
# ── CELL 9: Try Your Own Query ────────────────────────────────────────────────

research("Impact of AI on software engineering jobs")


 QUERY: Impact of AI on software engineering jobs

[1/2] Searching the web...
    Retrieved 5 sources.
[2/2] Synthesizing answer...

────────────────────────────────────────────────────────────
 RESEARCH ANSWER
────────────────────────────────────────────────────────────

The impact of AI on software engineering jobs is a topic of significant interest and debate. According to various sources, AI is transforming the software engineering profession, driving advancements that will reshape the industry [1], [2], [3]. While some experts believe that AI will replace software engineers, others argue that AI will augment their work, enabling them to focus on more complex and creative aspects of software design [3].

One of the primary effects of AI on software engineering is the automation of repetitive tasks, such as code generation [2], [5]. This shift is creating new opportunities for software engineers to focus on higher-level tasks, such as architecture, specifications, verification, and

{'query': 'Impact of AI on software engineering jobs',
 'web_results': "[1] The Impact of AI on Engineering Jobs\n    URL: https://www.intuit.com/blog/innovative-thinking/ai-impact-engineering-jobs\n    [Skip to main content](https://www.intuit.com/blog/innovative-thinking/ai-impact-engineering-jobs#content). *   [Share on LinkedIn](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.intuit.com%2Fblog%2Finnovative-thinking%2Fai-impact-engineering-jobs%2F&title=The+Impact+of+AI+on+Engineering+Jobs). *   [Print](https://www.intuit.com/blog/innovative-thinking/ai-impact-engineering-jobs#). If you’re looking to build or advance a career in this space, Intuit offers a range of [software engineering opportunities](https://www.intuit.com/careers/teams/software-engineering/), from backend software engineering to AI scientist roles. [Learning AI](https://www.intuit.com/blog/innovative-thinking/how-to-learn-artificial-intelligence/) and building hands-on experience are some of 